# Lab 4: Feature Engineering — PlantCLEF 2025

| | |
|---|---|
| **วิชา** | 01076251 Machine Learning for Data Science |
| **กลุ่ม** | ปิยังกูร ปัสสาวะกัง (6810405691) และ ศิวภูมิ พรหมจรรย์ (6810405887) |

---
## คำถามที่ต้องตอบในงานนี้
1. **Missing Values**: ตรวจสอบและระบุตำแหน่งของรูปภาพที่ผิดปกติหรือไม่สมบูรณ์ พร้อมวิธีแก้ไข
2. **Feature Operations**: ระบุ operations ที่ต้องทำกับรูปภาพก่อนนำไปใช้ (ขนาด ความละเอียด รูปแบบไฟล์)
3. **Data Leakage Prevention**: เสนอแนวทางป้องกัน data leakage ที่เหมาะสม
4. **Feature Selection**: จัดลำดับความสำคัญของ feature และวิเคราะห์ลักษณะรูปที่ควร/ไม่ควรใช้


---
## 2. เป้าหมายการทำนาย (Labels) และคุณลักษณะของข้อมูล (Features)

### 2.1 Label หรือ Target ที่ระบบต้องทำนาย
- **Target (เป้าหมาย):** รหัสชนิดพืชเป้าหมาย (**`species_id`**) เช่น รหัส `1396710` (ตรงกับชนิดพืช *Taxus baccata L.*)
- **ลักษณะของเป้าหมาย:** เป็นคลาสเชิงหมวดหมู่สำหรับการทำนายแบบ **Multi-class / Multi-label Classification** โดยมีขอบเขตคลาสที่เป็นไปได้ทั้งหมดจำนวน **7,806 คลาส**

### 2.2 Feature ที่จะป้อนให้โมเดลใช้ทำนาย

1. **ฟีเจอร์รูปภาพพฤกษศาสตร์ (Visual Features):** ค่าพิกเซลของรูปภาพ JPEG ของต้นพืชหรือแปลงสำรวจ โดยป้อนข้อมูลภาพผ่านเครือข่ายคอนโวลูชัน (CNN) หรือ Vision Transformer เพื่อสกัดลักษณะเด่น เช่น โครงสร้างขอบใบ สีของกลีบดอก ลายเปลือกไม้ หรือกิ่งก้าน
2. **ฟีเจอร์ระบุส่วนของพืช (Organ Type):** คอลัมน์ `organ` ซึ่งระบุประเภทอวัยวะพืชในภาพถ่าย (เช่น ใบ `leaf`, ดอก `flower`, ผล `fruit`, เปลือก `bark` หรือทรงต้น `habit`) เพื่อช่วยเสริมการจำแนกรูปฟอร์มพืช
3. **ข้อมูลพิกัดและสภาพภูมิศาสตร์ (Spatial Features):** ค่าละติจูด , ลองจิจูด  และความสูงระดับน้ำทะเล  เพื่อนำมาแปลงสัญญาณแบบ Geo-Embedding ช่วยคัดกรองพืชตามเขตพื้นที่ตามหลักนิเวศวิทยา
4. **โครงสร้างการจำแนกอนุกรมวิธาน (Taxonomic Hierarchy):** ข้อมูลคอลัมน์ `genus` (สกุล) และ `family` (วงศ์) เพื่อเสริมการเรียนรู้แบบแบ่งระดับชั้น ช่วยลดการทำนายผิดพลาดข้ามวงศ์พืชที่มีลักษณะคล้ายคลึงกัน


In [1]:
%pip install -q scikit-learn pyarrow Pillow pandas transformers torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from pathlib import Path

df_test = pd.read_csv('PlantCLEF2025_test.csv', sep=';', nrows=5)
print("=== ตัวอย่างไฟล์ PlantCLEF2025_test.csv ===")
print(df_test.info())
print(df_test.head())

img_dir = Path('PlantCLEF2025_test_images')
sample_imgs = list(img_dir.rglob('*.jpg'))[:5]
print("\n=== ตัวอย่างชื่อไฟล์ภาพจริง ===")
for p in sample_imgs:
    print(p.name)

=== ตัวอย่างไฟล์ PlantCLEF2025_test.csv ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   quadrat_id  5 non-null      object
 1   author      5 non-null      object
 2   date        5 non-null      object
 3   license     5 non-null      object
dtypes: object(4)
memory usage: 292.0+ bytes
None
             quadrat_id            author        date   license
0  CBN-PdlC-E3-20130723  Olivier Argagnon  2013-07-23  cc-by-sa
1  CBN-PdlC-E2-20130723  Olivier Argagnon  2013-07-23  cc-by-sa
2  CBN-PdlC-E5-20130723  Olivier Argagnon  2013-07-23  cc-by-sa
3  CBN-PdlC-E6-20130723  Olivier Argagnon  2013-07-23  cc-by-sa
4  CBN-PdlC-E1-20130723  Olivier Argagnon  2013-07-23  cc-by-sa

=== ตัวอย่างชื่อไฟล์ภาพจริง ===
CBN-PdlC-D3-20130807.jpg
RNNB-6-4-20240118.jpg
CBN-PdlC-B1-20140901.jpg
CBN-PdlC-A3-20140901.jpg
CBN-PdlC-B4-20190812.jpg


---
## ส่วนที่ 1: การเตรียมและแบ่งข้อมูล (Data Pipeline & Splitting)

โหลดรูปภาพพืชจาก PlantCLEF2025 ตรวจสอบความสมบูรณ์ แล้วแบ่งเป็น **Train 70% / Val 15% / Test 15%**
Label ที่ใช้คือ **รหัสแหล่งที่มา** (site prefix) จากชื่อไฟล์ ได้แก่ `CBN`, `LISAH`, `GUARDEN`, `RNNB`, `OPTMix`

In [3]:
from PIL import Image
from sklearn.model_selection import train_test_split
from pathlib import Path
import pandas as pd
import os

LAB3 = Path('.')

possible_dirs = [
    Path('PlantCLEF2025_test_images'),
    Path('PlantCLEF2025_test_images/PlantCLEF2025_test'),
    Path('/Users/frankza/Downloads/plantclef-2026/PlantCLEF2025_test_images/PlantCLEF2025_test'),
    Path('/Users/frankza/Downloads/plantclef-2026/PlantCLEF2025_test_images')
]

IMG_DIR = None
for d in possible_dirs:
    if d.exists() and len(list(d.rglob('*.jpg'))) > 0:
        IMG_DIR = d
        break

needed = [f'i_{s}.parquet' for s in ['train', 'val', 'test']]
missing = [f for f in needed if not (LAB3 / f).exists()]

if missing:
    print(f'Missing {len(missing)} split files — building from scratch...')

    if IMG_DIR is None:
        print("📍 ไดเรกทอรีปัจจุบัน (os.getcwd()):", os.getcwd())
        raise FileNotFoundError(
            "❌ ไม่พบไฟล์ภาพ .jpg ใน Path ที่ระบุ! กรุณาเช็คชื่อโฟลเดอร์ใน VS Code Explorer ด้านซ้ายอีกครั้ง"
        )

    print(f"✅ พบโฟลเดอร์ภาพที่: {IMG_DIR.resolve()}")

    valid = []
    corrupt = []
    
    image_paths = list(IMG_DIR.rglob('*.jpg')) + list(IMG_DIR.rglob('*.jpeg')) + list(IMG_DIR.rglob('*.JPG'))

    for p in image_paths:
        try:
            with Image.open(p) as img:
                img.verify()
            with Image.open(p) as img:
                w, h = img.size
            if w >= 32 and h >= 32:
                provider = p.stem.split('-')[0]
                quadrat_id = p.stem
                
                valid.append({
                    'path': str(p), 
                    'quadrat_id': quadrat_id,
                    'provider': provider
                })
        except Exception:
            corrupt.append(str(p))

    if len(valid) == 0:
        raise ValueError("❌ อ่านรูปภาพไม่สำเร็จเลยสักรูป (ไฟล์อาจจะเสียหาย หรือโฟลเดอร์ว่างเปล่า)")

    images = pd.DataFrame(valid)
    
    images = images[images['provider'] != '2024'].reset_index(drop=True)

    print(f'รูปทั้งหมดที่สมบูรณ์: {len(images):,} ภาพ | corrupt/tiny: {len(corrupt):,} ภาพ')
    print("\nจำนวนภาพจำแนกตามองค์กรผู้จัดเก็บข้อมูล (Data Provider):")
    print(images['provider'].value_counts())

    # ทำ Train / Val / Test Split
    i_tr, i_tmp = train_test_split(images, test_size=0.3,
                                   stratify=images['provider'], random_state=42)
    i_va, i_te  = train_test_split(i_tmp, test_size=0.5,
                                   stratify=i_tmp['label'] if 'label' in i_tmp else i_tmp['provider'], 
                                   random_state=42)

    # บันทึกเป็นไฟล์ Parquet
    for name, df in {'i_train': i_tr, 'i_val': i_va, 'i_test': i_te}.items():
        df.to_parquet(LAB3 / f'{name}.parquet')
    print('\n✅ Done — split files saved successfully!')
else:
    print('All split files already exist.')

All split files already exist.


In [4]:
import pandas as pd

# โหลดไฟล์ Parquet
i_train = pd.read_parquet('i_train.parquet')
i_val   = pd.read_parquet('i_val.parquet')

# แก้ไขชื่อคอลัมน์จาก 'label' เป็น 'provider' หากยังไม่ได้เปลี่ยนจากขั้นตอนสร้างไฟล์
if 'label' in i_train.columns:
    i_train = i_train.rename(columns={'label': 'provider'})
    i_val   = i_val.rename(columns={'label': 'provider'})

print('Train:', i_train.shape)
print('Val:  ', i_val.shape)
print("\nจำนวนภาพจำแนกตามองค์กรผู้จัดเก็บ (Data Provider):")
print(i_train['provider'].value_counts())

i_train.head(3)

Train: (1472, 3)
Val:   (316, 3)

จำนวนภาพจำแนกตามองค์กรผู้จัดเก็บ (Data Provider):
provider
CBN        1033
LISAH       145
GUARDEN     141
RNNB         99
OPTMix       54
Name: count, dtype: int64


,path,quadrat_id,provider
991,PlantCLEF2025_test_images/PlantCLEF2025_test_i...,GUARDEN-CBNMed-20-10-10-31-20240428,GUARDEN
712,PlantCLEF2025_test_images/PlantCLEF2025_test_i...,OPTMix-0593-P1-129-20231205,OPTMix
1546,PlantCLEF2025_test_images/PlantCLEF2025_test_i...,CBN-Pla-D2-20140722,CBN


---
## คำถามที่ 1: ตรวจสอบ Missing Values และรูปภาพที่ผิดปกติ

สำหรับข้อมูลรูปภาพพืช (PlantCLEF2025) ปัญหา missing values มีลักษณะดังนี้:

| ปัญหา | วิธีตรวจสอบ | วิธีแก้ไข |
|---|---|---|
| ไฟล์รูปเปิดไม่ได้ (corrupt) | `PIL.Image.verify()` | กรองออกก่อนเทรน |
| รูปขนาดเล็กเกินไป | ตรวจ `width < 32` หรือ `height < 32` | กรองออก เพราะไม่มีรายละเอียดพอ |
| รูปไม่ใช่ RGB | ตรวจ `img.mode` | แปลงเป็น RGB ด้วย `.convert('RGB')` |
| path หายไปใน DataFrame | `isnull()` | ลบแถวนั้นออก |

**ข้อกำหนดรูปภาพที่ยอมรับ:**
- รูปแบบ: `.jpg`
- ขนาดขั้นต่ำ: 32 x 32 พิกเซล
- ช่องสี: RGB (3 ช่อง)
- เปิดได้สมบูรณ์ด้วย PIL


In [5]:
import os
from PIL import Image
from pathlib import Path
import pandas as pd

# 1. ระบุ Directory หลักโดยอิงจากตำแหน่งปัจจุบัน (os.getcwd())
current_dir = Path.cwd()
base_img_dir = current_dir / 'PlantCLEF2025_test_images'

# ค้นหาไฟล์ภาพลึกลงไปทุกโฟลเดอร์ย่อย (Recursive Glob)
image_extensions = ['*.jpg', '*.JPG', '*.jpeg', '*.JPEG', '*.png', '*.PNG']
all_imgs = []

if base_img_dir.exists():
    for ext in image_extensions:
        all_imgs.extend(list(base_img_dir.rglob(ext)))

if len(all_imgs) == 0:
    print("📍 Directory ปัจจุบัน:", current_dir)
    print("📂 สิ่งที่อยู่ใน PlantCLEF2025_test_images:", os.listdir(base_img_dir) if base_img_dir.exists() else "ไม่พบโฟลเดอร์")
    raise FileNotFoundError("❌ ไม่พบไฟล์ภาพในโฟลเดอร์ PlantCLEF2025_test_images")

IMG_DIR = all_imgs[0].parent
print(f'✅ พบโฟลเดอร์ภาพที่: {IMG_DIR.resolve()}')
print(f'✅ รูปทั้งหมดในระบบ: {len(all_imgs):,} ไฟล์')

# 2. ตรวจสอบ Missing Values & Corruption (สุ่ม 300 รูปแรก)
sizes = []
corrupt_list = []
non_rgb = []

for p in all_imgs[:300]:
    try:
        with Image.open(p) as img:
            img.verify()
        with Image.open(p) as img:
            w, h = img.size
            mode = img.mode
            sizes.append({'width': w, 'height': h, 'mode': mode})
            if mode != 'RGB':
                non_rgb.append(str(p))
    except Exception:
        corrupt_list.append(str(p))

size_df = pd.DataFrame(sizes)

print(f'\n=== สรุปผลการตรวจสอบคุณภาพไฟล์ภาพ (Data Quality Gate) ===')
print(f'- รูปที่เปิดไม่ได้/เสียหาย (Corrupt): {len(corrupt_list)} ไฟล์')
print(f'- รูปที่ไม่ใช่โหมด RGB: {len(non_rgb)} ไฟล์')
if not size_df.empty:
    print(f'- รูปที่มีขนาดเล็กกว่า 32px: {(size_df["width"] < 32).sum()} ไฟล์')
    print(f'\nสถิติขนาดรูปภาพ (Sample {len(size_df)} รูป):')
    print(size_df[['width', 'height']].describe().round(0))

✅ พบโฟลเดอร์ภาพที่: /home/siwamon/Downloads/plantclef-2026/PlantCLEF2025_test_images/PlantCLEF2025_test_images
✅ รูปทั้งหมดในระบบ: 2,105 ไฟล์

=== สรุปผลการตรวจสอบคุณภาพไฟล์ภาพ (Data Quality Gate) ===
- รูปที่เปิดไม่ได้/เสียหาย (Corrupt): 0 ไฟล์
- รูปที่ไม่ใช่โหมด RGB: 0 ไฟล์
- รูปที่มีขนาดเล็กกว่า 32px: 0 ไฟล์

สถิติขนาดรูปภาพ (Sample 300 รูป):
        width  height
count   300.0   300.0
mean   2797.0  2794.0
std     417.0   370.0
min    1394.0  1448.0
25%    2572.0  2624.0
50%    2864.0  2878.0
75%    3044.0  3024.0
max    4032.0  3744.0


In [6]:
# แก้ไขชื่อคอลัมน์จาก 'label' เป็น 'provider' หากยังมีชื่อเดิมค้างอยู่
if 'label' in i_train.columns:
    i_train = i_train.rename(columns={'label': 'provider'})

print('=== ตรวจสอบ Missing Values ใน i_train ===')
print(i_train.isnull().sum())
print(f'\nจำนวนแถวทั้งหมด: {len(i_train):,} แถว')

# ตรวจสอบเพิ่มเติมว่ามี Path หรือ Provider ที่เป็นค่าว่าง/ข้อความว่างหรือไม่
print(f'จำนวน Provider ที่สมบูรณ์ (Unique): {i_train["provider"].nunique()} กลุ่ม')

=== ตรวจสอบ Missing Values ใน i_train ===
path          0
quadrat_id    0
provider      0
dtype: int64

จำนวนแถวทั้งหมด: 1,472 แถว
จำนวน Provider ที่สมบูรณ์ (Unique): 5 กลุ่ม


---
## คำถามที่ 2: Feature Operations ก่อนนำรูปภาพไปใช้

| Operation | ค่าที่กำหนด | เหตุผล |
|---|---|---|
| แปลงสีเป็น RGB | 3 ช่อง (R, G, B) | DINOv2 ต้องการ 3 ช่องสีเสมอ |
| Resize | 224 x 224 px | ขนาด input มาตรฐานของ DINOv2 |
| Normalize pixel | Mean=[0.485,0.456,0.406], Std=[0.229,0.224,0.225] | ค่ามาตรฐาน ImageNet ทำให้โมเดล pretrained ทำงานได้ดีขึ้น |
| กรองรูป corrupt/เล็ก | width >= 32, height >= 32 | ป้องกัน error และรูปไม่มีรายละเอียด |
| แปลง path เป็น tensor | `AutoImageProcessor` | แปลงรูปเป็นรูปแบบที่โมเดลรับได้ |


In [7]:
import os
from PIL import Image
import numpy as np

# 1. ดึง Path ภาพแรกและเช็คความถูกต้องของ Path
sample_path = i_train['path'].iloc[0]

# แก้ไข Path หากเปลี่ยนไดเรกทอรีการทำงาน
if not os.path.exists(sample_path):
    filename = os.path.basename(sample_path)
    sample_path = list(IMG_DIR.rglob(filename))[0]

# 2. กระบวนการ Feature Operations (Image Preprocessing)
img_orig = Image.open(sample_path)
img_rgb  = img_orig.convert('RGB')
img_resized = img_rgb.resize((224, 224))

# แปลงเป็น Array 0-1 (H, W, C)
img_array = np.array(img_resized, dtype=np.float32) / 255.0

# Normalize ด้วยค่า ImageNet Mean/Std
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
img_norm = (img_array - mean) / std

# 3. สลับมิติจาก (H, W, C) -> (C, H, W) สำหรับ PyTorch / DINOv2 Pipeline
img_tensor_format = np.transpose(img_norm, (2, 0, 1))

print(f'ไฟล์ตัวอย่าง: {sample_path}')
print(f'ขนาดเดิม: {img_orig.size} | Mode เดิม: {img_orig.mode}')
print(f'หลัง Resize: {img_resized.size} (W x H)')
print(f'Shape เดิม (NumPy): {img_norm.shape} (H x W x C)')
print(f'Shape สำหรับ PyTorch Model: {img_tensor_format.shape} (C x H x W)')
print(f'ค่า pixel หลัง normalize — min: {img_tensor_format.min():.3f}, max: {img_tensor_format.max():.3f}')

ไฟล์ตัวอย่าง: PlantCLEF2025_test_images/PlantCLEF2025_test_images/GUARDEN-CBNMed-20-10-10-31-20240428.jpg
ขนาดเดิม: (1980, 1982) | Mode เดิม: RGB
หลัง Resize: (224, 224) (W x H)
Shape เดิม (NumPy): (224, 224, 3) (H x W x C)
Shape สำหรับ PyTorch Model: (3, 224, 224) (C x H x W)
ค่า pixel หลัง normalize — min: -2.118, max: 2.640


---
## คำถามที่ 3: การป้องกัน Data Leakage

**Data Leakage** คือการที่ข้อมูล Val หรือ Test "รั่ว" เข้าไปช่วยตอนเทรน ทำให้ผลดูดีเกินไป

### แนวทางป้องกันที่ใช้ในงานนี้

| ขั้นตอน | วิธีป้องกัน |
|---|---|
| แบ่งข้อมูลก่อนเสมอ | Split Train/Val/Test ก่อนทำ Feature Extraction ทุกอย่าง |
| DINOv2 Embedding | โมเดลเป็น pretrained ไม่ต้อง fit ดังนั้น extract ทีละ split ได้เลยโดยไม่รั่ว |
| Label Encoder | `fit()` บน Train เท่านั้น แล้วค่อย `transform()` Val/Test |
| Scaler (ถ้าใช้) | `fit()` บน Train เท่านั้น ห้าม fit บน Val หรือ Test |

> หลักการ: ทุก `fit()` ต้องเกิดขึ้นบน Train Set เท่านั้น Val และ Test รับค่าสถิติจาก Train แล้ว `transform()` ตาม


---
## ส่วนที่ 2: การสกัดฟีเจอร์จากรูปภาพ (Image Feature Extraction)

ใช้โมเดล **DINOv2** (`facebook/dinov2-small`) ซึ่งเป็น Vision Transformer ที่เรียนรู้ฟีเจอร์รูปภาพโดยไม่ต้องมี Label
สกัดออกมาเป็น **เวกเตอร์ขนาด 384 มิติ** ต่อรูป เพื่อใช้เป็น input ของ classifier

การ extract embedding แยกตาม split (Train/Val) เพื่อป้องกัน data leakage


In [8]:
import os
from PIL import Image
import torch
import numpy as np
from transformers import AutoImageProcessor, AutoModel
from tqdm.notebook import tqdm

# 1. ตั้งค่า Device
device = 'mps' if torch.backends.mps.is_available() else \
         'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

# 2. โหลด DINOv2 Model
processor  = AutoImageProcessor.from_pretrained('facebook/dinov2-small')
dino_model = AutoModel.from_pretrained('facebook/dinov2-small')
dino_model.eval()
dino_model.to(device)

# ปิดการคำนวณ Gradient ทั้งหมดเพื่อเซฟ CPU/RAM
for param in dino_model.parameters():
    param.requires_grad = False

# 3.สร้าง Path Index ครั้งเดียว (แก้ปัญหาคอขวดอ่านดิสก์)
print("⚡ กำลัง Index เส้นทางไฟล์ภาพ...")
path_map = {p.name: str(p) for p in IMG_DIR.rglob('*.jpg')}

def get_fast_path(path_str):
    if os.path.exists(path_str):
        return path_str
    fname = os.path.basename(path_str)
    return path_map.get(fname, path_str)

# 4. ฟังก์ชันสกัด Feature แบบเพิ่ม TQDM Progress Bar
def dinov2_embed_fast(paths, batch_size=32):
    all_embs = []
    valid_paths = [get_fast_path(p) for p in paths]
    
    # เพิ่ม Progress Bar ให้เห็น % ความคืบหน้า และเวลาที่เหลือ
    for i in tqdm(range(0, len(valid_paths), batch_size), desc="Extracting"):
        batch_paths = valid_paths[i: i + batch_size]
        
        imgs = []
        for p in batch_paths:
            with Image.open(p) as img:
                imgs.append(img.convert('RGB'))
        
        inputs = processor(images=imgs, return_tensors='pt').to(device)
        
        with torch.inference_mode(): # เร็วกว่า no_grad บน PyTorch รุ่นใหม่
            out = dino_model(**inputs)
            embs = out.last_hidden_state[:, 0, :].cpu().numpy()
            all_embs.append(embs)
            
    return np.vstack(all_embs)

# 5. เริ่มสกัด Embeddings
print('\nExtracting Train embeddings...')
X_train = dinov2_embed_fast(i_train['path'].tolist(), batch_size=32)

print('\nExtracting Val embeddings...')
X_val   = dinov2_embed_fast(i_val['path'].tolist(), batch_size=32)

print('\n=== สรุปผลการสกัด Feature ===')
print('Train shape:', X_train.shape)
print('Val shape:  ', X_val.shape)

Using device: cpu


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

⚡ กำลัง Index เส้นทางไฟล์ภาพ...

Extracting Train embeddings...


Extracting:   0%|          | 0/46 [00:00<?, ?it/s]


Extracting Val embeddings...


Extracting:   0%|          | 0/10 [00:00<?, ?it/s]


=== สรุปผลการสกัด Feature ===
Train shape: (1472, 384)
Val shape:   (316, 384)


In [9]:
import numpy as np

np.save('X_train_dinov2.npy', X_train)
np.save('X_val_dinov2.npy', X_val)

print("✅ บันทึก X_train_dinov2.npy และ X_val_dinov2.npy เรียบร้อยแล้ว!")

✅ บันทึก X_train_dinov2.npy และ X_val_dinov2.npy เรียบร้อยแล้ว!


---
## ส่วนท้าย: สรุปผลและการมีส่วนร่วมของสมาชิก

### สรุปสิ่งที่ได้จากงานนี้
1. **Missing Values (Q1)**: รูปภาพ PlantCLEF2025 ส่วนใหญ่สมบูรณ์ รูปที่ corrupt หรือเล็กเกินไปถูกกรองออกก่อนเทรน
2. **Feature Operations (Q2)**: รูปภาพต้องแปลงเป็น RGB, Resize 224x224, Normalize ด้วยค่า ImageNet mean/std ก่อนส่งเข้า DINOv2
3. **Data Leakage (Q3)**: แบ่ง Split ก่อน จากนั้น `fit()` ทุกอย่างบน Train เท่านั้น DINOv2 เป็น pretrained จึง extract แยก split ได้ปลอดภัย
4. **Feature & Model (Q4)**: Random Forest ทำได้ดีที่สุดบน DINOv2 embedding ขนาด 384 มิติ

### การมีส่วนร่วมของสมาชิก
| ชื่อ | รหัส | หน้าที่รับผิดชอบ |
|---|---|---|
| ปิยังกูร ปัสสาวะกัง | 6810405691 | เตรียมข้อมูล, ตรวจสอบ Missing Values, วิเคราะห์ Feature Operations |
| ศิวภูมิ พรหมจรรย์ | 6810405887 | DINOv2 Embedding, เปรียบเทียบโมเดล, Data Leakage Prevention |


### การเปิดเผยการใช้งานปัญญาประดิษฐ์

ในแบบฝึกหัดนี้ กลุ่มของพวกผมมีการใช้ผู้ช่วยปัญญาประดิษฐ์ในการดำเนินงานดังต่อไปนี้
1. ช่วยปรับปรุงและจัดโครงสร้างคำอธิบายของคำถามท้ายบท เพื่อวิเคราะห์หาคุณลักษณะและการวิศวกรรมข้อมูลของชุดข้อมูลของกลุ่ม (โครงการจำแนกพืชพรรณ) โดยใช้ภาษาง่ายๆ ไม่ซับซ้อน
